In [1]:
from collections import defaultdict
import datetime
import json
with open("odds_matches.json") as f:
    matches = json.load(f)
bookies = defaultdict(list)
BOOKIES = {
    "all": "average",
    "163": "eFortuna",
    "165": "STS",
    "502": "LV BET",
    "572": "BETFAN",
    "591": "Superbet",
}
all_bookies = set()
def probablity_home_win(home_odds, away_odds):
    ph_raw = 1.0 / float(home_odds)
    pa_raw = 1.0 / float(away_odds)

    total = ph_raw + pa_raw

    return ph_raw/total
matches = sorted(matches, key=lambda x: x["date-start-base"])
for match in matches:
    home_win = int(match["home-winner"] == "win")
    name = match["name"]
    home_score = int(match["homeResult"])
    away_score = int(match["awayResult"])
    # date from seconds since epoch
    match_date = datetime.datetime.fromtimestamp(match["date-start-base"]).date()
    odds = match["odds"]
    if len(odds) < 1:
        print(odds)
        continue
    home_odds, away_odds = odds
    
    if match_date < datetime.date(2025, 1, 1):
        continue
    
    bookies["all"].append({
        "bookie": "average",
        "home_win": home_win,
        "home_odds": home_odds["avgOdds"],
        "away_odds": away_odds["avgOdds"],
        "BoN": max(home_score, away_score) * 2 - 1,
        "home_prob": probablity_home_win(home_odds["avgOdds"], away_odds["avgOdds"]),
        "name": name
    })
    for provider, odds in match["openingsOdds"].items():
        home, away = odds
        if not (float(home) and float(away)):
            continue
        
        all_bookies.add(provider)
        if provider in BOOKIES:
            bookies[provider].append({
                "bookie": BOOKIES[provider],
                "home_win": home_win,
                "home_odds": float(home),
                "away_odds": float(away),
                "BoN": max(home_score, away_score) * 2 - 1,
                "home_prob": probablity_home_win(home, away),
                "name": name
            })

print(len(all_bookies))

[]
[]
[]
[]
5


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score
)
import pandas as pd

for provider, records in bookies.items():
    df = pd.DataFrame(records)
    X = df["home_prob"]
    y = df["home_win"]
    size = len(X)
    auc = roc_auc_score(y, X)
    ll = log_loss(y, X)
    brier_score = brier_score_loss(y, X)
    name = BOOKIES[provider]

    print(f"{name}(size={size}): AUC={auc:.3f} Log Loss={ll:.3f} Brier Score={brier_score:.3f}")
    for N in [1, 3, 5]:
        bon = df.loc[df["BoN"] == N]
        size = len(bon)
        X = bon["home_prob"]
        y = bon["home_win"]
        auc = roc_auc_score(y, X)
        ll = log_loss(y, X)
        brier_score = brier_score_loss(y, X)
        print(f"Bo{N} (size={size}) AUC={auc:.3f} Log Loss={ll:.3f} Brier Score={brier_score:.3f}")

average(size=872): AUC=0.800 Log Loss=0.546
Bo1 (size=172) AUC=0.792 Log Loss=0.565 Brier Score=0.191
Bo3 (size=521) AUC=0.812 Log Loss=0.535 Brier Score=0.178
Bo5 (size=179) AUC=0.788 Log Loss=0.562 Brier Score=0.191
eFortuna(size=562): AUC=0.779 Log Loss=0.563
Bo1 (size=148) AUC=0.771 Log Loss=0.581 Brier Score=0.198
Bo3 (size=326) AUC=0.796 Log Loss=0.548 Brier Score=0.184
Bo5 (size=88) AUC=0.750 Log Loss=0.586 Brier Score=0.205
BETFAN(size=832): AUC=0.796 Log Loss=0.550
Bo1 (size=168) AUC=0.774 Log Loss=0.579 Brier Score=0.197
Bo3 (size=511) AUC=0.814 Log Loss=0.534 Brier Score=0.178
Bo5 (size=153) AUC=0.765 Log Loss=0.571 Brier Score=0.197
Superbet(size=830): AUC=0.795 Log Loss=0.552
Bo1 (size=167) AUC=0.786 Log Loss=0.572 Brier Score=0.193
Bo3 (size=510) AUC=0.813 Log Loss=0.536 Brier Score=0.178
Bo5 (size=153) AUC=0.757 Log Loss=0.583 Brier Score=0.201
STS(size=835): AUC=0.793 Log Loss=0.553
Bo1 (size=172) AUC=0.774 Log Loss=0.578 Brier Score=0.196
Bo3 (size=513) AUC=0.810 Log L